<a href="https://colab.research.google.com/github/Girinath-Codes/Weather_Agent/blob/Develop/Weather_Forecast_Agent_with_Pydantic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install the pydantic library**

In [7]:
!pip install pydantic-ai==0.1.3

In [8]:
import nest_asyncio

nest_asyncio.apply()

In [9]:
import requests

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
API_KEY = "0f98a6b3ca2b46209b7b3df53b13922d"

# Custom weather forcast function
def find_weather(city:str) -> dict:
  """This function returns the current weather forecast for the given city"""
  units = "metrics"
  params = {
      'q' : city,
      'appid' : API_KEY,
      'units' : units
  }

  response = requests.get(BASE_URL, params=params)
  result = response.json()

  return result

In [10]:
op = find_weather("chennai")
print(op)

{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}


# **Pydantic Agent**

In [12]:
import os
import requests
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from pydantic_ai.settings import ModelSettings

In [14]:
os.environ["GROQ_API_KEY"] = "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"

In [15]:
#Define the output schema
class WeatherForecast(BaseModel):
  location: str
  description : str
  temperature_celsius : float

In [20]:
weather_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    model_settings=ModelSettings(temperature=0.2),
    output_type=str,
    system_prompt=("You are a helpful weather assistant. use the 'get_weather_forecast'tool to find current weather"
        "conditions for any city. Provide clean and friendly answers.")
)


In [21]:
# weather forecast tool
@weather_agent.tool
def get_weather_forecast(ctx: RunContext, city: str) -> WeatherForecast:

  url = "https://api.openweathermap.org/data/2.5/weather"
  api_key = "0f98a6b3ca2b46209b7b3df53b13922d"
  params = {
      'q' : city,
      'appid' : api_key,
      'units' : 'metrics'
  }

  res = requests.get(url, params=params).json()


  return WeatherForecast(
      location=res['name'],
      description=res['weather'][0]['description'].capitalize(),
      temperature_celsius=res['main']['temp']
  )

In [28]:
question = input("Enter your weather: ")
result = weather_agent.run_sync(question)
print("\nForecast:", result.output)

Enter your weather: chennai

Forecast: It seems like the weather in Chennai is currently hazy with a temperature of 30.83°C.
